In [1]:
import pandas as pd
import re

archivo_recesa = "Ventas redes RECESA.xlsx"
archivo_redelsa = "Ventas redes REDELSA.xlsx"

In [2]:
excel_recesa = pd.ExcelFile(archivo_recesa)
excel_redelsa = pd.ExcelFile(archivo_redelsa)

print("Hojas RECESA:", excel_recesa.sheet_names)
print("Hojas REDELSA:", excel_redelsa.sheet_names)

Hojas RECESA: ['Ventas_Recesa']
Hojas REDELSA: ['Ventas_Redelsa']


In [3]:
df_recesa_raw = pd.read_excel(
    archivo_recesa,
    sheet_name="Ventas_Recesa",
    header=None
)

df_redelsa_raw = pd.read_excel(
    archivo_redelsa,
    sheet_name="Ventas_Redelsa",
    header=None
)

df_recesa_raw.head(10)

,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
0,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,septiembre 2025,NaN,NaN,octubre 2025,NaN,NaN,noviembre 2025,NaN,NaN,...,NaN,agosto 2026,NaN,NaN,septiembre 2026,NaN,NaN,NaN,NaN,NaN
2,NaN,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,...,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio
3,Total,700,8,78.125,4910,378,11.597698,4357.12,252,15.437579,...,1.327079,20893.14,855,21.818246,5829.4,167,31.166647,246697.58,225183,0.978163
4,PRODUCTO TERMINADO / HOJAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,283.04,NaN,NaN,NaN,3100,1,2767.86,3417,2,1525.45
5,[10012020-A] SUZUKI SAMURAY TRA 1a 1...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,283.04,NaN,NaN,NaN,NaN,NaN,NaN,317,1,283.04
6,[69012641-M] HENDRICKSON AIR TEK DEL...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3100,1,2767.86,3100,1,2767.86
7,PRODUCTO TERMINADO / PERNO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1550,100,13.8393
8,[BR383] BARRA ROSCADA GALV 3/8 X 3MT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1550,100,13.8393
9,PRODUCTO TERMINADO / PERNO ESTRUCTURAL,640,4,142.8575,2717.5,100,24.2634,3984,20,177.857,...,NaN,14940,50,266.7858,965.6,8,107.7675,120695.5,1042,103.420192


In [4]:
df_redelsa_raw.head(10)

,0,1,2,3,4,5,6,7,8,9,...,45,46,47,48,49,50,51,52,53,54
0,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,mayo 2025,NaN,NaN,junio 2025,NaN,NaN,julio 2025,NaN,NaN,...,NaN,agosto 2026,NaN,NaN,septiembre 2026,NaN,NaN,NaN,NaN,NaN
2,NaN,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,...,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio
3,Total,257.2,8,28.70625,16993.88,224,67.737098,64494.73,243,236.973498,...,5.785758,136593.57,14153,8.617155,205127.49,19198,9.540035,1822247.44,184850,8.801767
4,PRODUCTO TERMINADO / BALANCIN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,571.14,2,254.975
5,[BHD-0101] BALANCIN P/RESORTAJE HINO...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,571.14,2,254.975
6,PRODUCTO TERMINADO / HOJAS,NaN,NaN,NaN,7863.19,26,270.027308,11965.12,35,305.232286,...,290.503469,9450,25,337.5,12643.85,47,240.194681,329099.99,1141,257.527853
7,[10011317-M] CARRETON TRA 1a 12 1/2X...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,452.68,2,202.09
8,[10012020-A] SUZUKI SAMURAY TRA 1a 1...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,235.32,1,210.11
9,[10012020-O] CHEVROLET CMV TRA 1a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1199.01,4,267.6375


In [5]:
import pandas as pd
import re

def limpiar_texto(texto):
    texto = str(texto).strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto


def extraer_categoria(texto):
    """
    Extrae la categoría desde filas tipo:

    PRODUCTO TERMINADO / PERNO
    -> PERNO

    REPUESTOS Y ACCESORIOS / ROLDANA / TORNILLERIA
    -> ROLDANA Y TORNILLERIA

    Regla:
    - Ignora el primer texto antes del primer "/".
    - Toma todos los textos después del primer "/".
    - Los une con " Y ".
    """
    texto = limpiar_texto(texto)

    if "/" not in texto:
        return None

    partes = [limpiar_texto(p) for p in texto.split("/") if limpiar_texto(p)]

    if len(partes) >= 2:
        return " Y ".join(partes[1:]).upper()

    return None


def transformar_redes(df_raw, empresa):

    filas = []

    data = df_raw.iloc[4:, :].copy()

    categoria_actual = "SIN CATEGORIA"
    categorias = []

    for valor in data[0]:
        texto = limpiar_texto(valor)

        if not re.search(r"\[.*?\]", texto):
            categoria_detectada = extraer_categoria(texto)

            if categoria_detectada:
                categoria_actual = categoria_detectada

            categorias.append(None)

        else:
            categorias.append(categoria_actual)

    data["Categoria"] = categorias

    data["Codigo"] = data[0].astype(str).str.extract(r"\[(.*?)\]")

    data["Descripcion"] = data[0].astype(str).str.replace(
        r"\[.*?\]\s*",
        "",
        regex=True
    )

    data["Descripcion"] = data["Descripcion"].apply(limpiar_texto)

    data = data.dropna(subset=["Codigo"]).copy()

    for col in range(1, df_raw.shape[1], 3):

        mes = df_raw.iloc[1, col]

        if pd.isna(mes):
            continue

        temp = pd.DataFrame({
            "Empresa": empresa,
            "Codigo": data["Codigo"],
            "Descripcion": data["Descripcion"],
            "Categoria": data["Categoria"],
            "Mes": mes,
            "Total": data[col],
            "Cantidad": data[col + 1],
            "Precio_Promedio": data[col + 2]
        })

        filas.append(temp)

    df_final = pd.concat(filas, ignore_index=True)

    for columna in ["Total", "Cantidad", "Precio_Promedio"]:
        df_final[columna] = pd.to_numeric(df_final[columna], errors="coerce").fillna(0)

    df_final["Año"] = df_final["Mes"].astype(str).str.extract(r"(\d{4})").astype(int)

    df_final = df_final[
        (df_final["Total"] != 0) |
        (df_final["Cantidad"] != 0)
    ].copy()

    return df_final


In [6]:
df_recesa = transformar_redes(df_recesa_raw, "RECESA")
df_redelsa = transformar_redes(df_redelsa_raw, "REDELSA")

In [7]:
df_total = pd.concat(
    [df_recesa, df_redelsa],
    ignore_index=True
)

df_total.head(20)

,Empresa,Codigo,Descripcion,Categoria,Mes,Total,Cantidad,Precio_Promedio,Año
0,RECESA,PEA-1912,PERNO DE ANCLAJE R.O.G.5 1 X 9 X 3 TIPO J,PERNO ESTRUCTURAL,septiembre 2025,640.00,4.0,142.857500,2025
1,RECESA,TRO51,TUERCA HEX RO5 1,ROLDANA Y TORNILLERIA,septiembre 2025,60.00,4.0,13.392500,2025
2,RECESA,PSM,PERNO S/MUESTRA,PERNO ESTRUCTURAL,octubre 2025,2717.50,100.0,24.263400,2025
3,RECESA,RPF43612,"ROLDANA F436 1/2""",ROLDANA Y TORNILLERIA,octubre 2025,17.50,50.0,0.312600,2025
4,RECESA,TROA325112,"TUERCA HEX. A325 1 1/2""",ROLDANA Y TORNILLERIA,octubre 2025,1104.00,64.0,15.401719,2025
5,RECESA,TROA325114,TUERCA HEX. A325 1.1/4,ROLDANA Y TORNILLERIA,octubre 2025,976.00,64.0,13.616094,2025
6,RECESA,TROA32512,TUERCA HEX. A325 1/2,ROLDANA Y TORNILLERIA,octubre 2025,95.00,100.0,0.848200,2025
7,RECESA,PSM,PERNO S/MUESTRA,PERNO ESTRUCTURAL,noviembre 2025,3984.00,20.0,177.857000,2025
8,RECESA,ROA32512112,TORNILLO HEX. A325 1/2 X 1.1/2,ROLDANA Y TORNILLERIA,noviembre 2025,32.00,16.0,1.785625,2025
9,RECESA,ROA32558212,TORNILLO A325 5/8 X 2 1/2,ROLDANA Y TORNILLERIA,noviembre 2025,80.00,32.0,2.232188,2025


In [8]:
resumen_empresa = (
    df_total
    .groupby("Empresa", as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
)

resumen_empresa["Precio_Promedio_Global"] = (
    resumen_empresa["Total"] / resumen_empresa["Cantidad"]
)

resumen_empresa

,Empresa,Total,Cantidad,Precio_Promedio_Global
0,RECESA,246697.58,225183.0,1.095543
1,REDELSA,1822247.44,184850.0,9.857979


In [9]:
ventas_mes_empresa = (
    df_total
    .groupby(["Empresa", "Año", "Mes"], as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
)

ventas_mes_empresa["Precio_Promedio_Global"] = (
    ventas_mes_empresa["Total"] / ventas_mes_empresa["Cantidad"]
)

ventas_mes_empresa

,Empresa,Año,Mes,Total,Cantidad,Precio_Promedio_Global
0,RECESA,2025,noviembre 2025,4357.12,252.0,17.290159
1,RECESA,2025,octubre 2025,4910.00,378.0,12.989418
2,RECESA,2025,septiembre 2025,700.00,8.0,87.500000
3,RECESA,2026,abril 2026,121814.70,44215.0,2.755054
4,RECESA,2026,agosto 2026,20893.14,855.0,24.436421
5,RECESA,2026,febrero 2026,10115.00,31900.0,0.317085
6,RECESA,2026,junio 2026,5280.92,3553.0,1.486327
7,RECESA,2026,marzo 2026,40038.10,134875.0,0.296853
8,RECESA,2026,mayo 2026,32759.20,8980.0,3.648018
9,RECESA,2026,septiembre 2026,5829.40,167.0,34.906587


In [10]:
top_productos_ventas = (
    df_total
    .groupby(["Empresa", "Codigo", "Descripcion"], as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
    .sort_values("Total", ascending=False)
)

top_productos_ventas.head(20)

,Empresa,Codigo,Descripcion,Total,Cantidad
408,REDELSA,PSM,PERNO S/MUESTRA,563945.68,11036.0
6,RECESA,PSM,PERNO S/MUESTRA,120055.50,1038.0
235,REDELSA,BRI-25355,BARRA LISA F1554/GR55 25MM X 3.55 MTS,45865.17,143.0
489,REDELSA,ROA325582,"TORNILLO A325 5/8 X 2""",40963.20,10540.0
646,REDELSA,TROA32558,TUERCA HEX. A325 5/8,38096.70,25186.0
100,REDELSA,21032028-M,TOYOTA HILUX MOD TRA 3a 20X28 REFUERZO,36716.96,189.0
490,REDELSA,ROA32558212,TORNILLO A325 5/8 X 2 1/2,33448.32,8164.0
428,REDELSA,RC-55-148,RESORTE C INTER DEL. 55-148 9HN 4X7/16 (S4000),33333.50,16.0
482,REDELSA,ROA3251312,TORNILLO HEX. A325 1 X 3 1/2,33141.46,2146.0
640,REDELSA,TROA3251,"TUERCA HEX. A325 1""",31888.55,4925.0


In [11]:
top_productos_cantidad = (
    df_total
    .groupby(["Empresa", "Codigo", "Descripcion"], as_index=False)
    .agg({
        "Cantidad": "sum",
        "Total": "sum"
    })
    .sort_values("Cantidad", ascending=False)
)

top_productos_cantidad.head(20)

,Empresa,Codigo,Descripcion,Cantidad,Total
35,RECESA,TPL14112,TORNILLO LAMINA 14 X 1 1/2,134520.0,30939.60
512,REDELSA,RPF43658,ROLDANA F436 5/8,27482.0,17969.08
646,REDELSA,TROA32558,TUERCA HEX. A325 5/8,25186.0,38096.70
51,RECESA,TSDT141,TORNILLO POLSER P/BROCA TROPICALIZADO1/4 X 1,24900.0,8715.00
408,REDELSA,PSM,PERNO S/MUESTRA,11036.0,563945.68
489,REDELSA,ROA325582,"TORNILLO A325 5/8 X 2""",10540.0,40963.20
645,REDELSA,TROA32534,TUERCA HEX. A325 3/4,10106.0,31454.30
509,REDELSA,RPF43612,"ROLDANA F436 1/2""",8797.0,3770.72
22,RECESA,RP38,ROLDANA 3/8,8580.0,1745.80
490,REDELSA,ROA32558212,TORNILLO A325 5/8 X 2 1/2,8164.0,33448.32


In [12]:
# PARTE 9 — Crear ventas corporativas por mes

ventas_corporativas = (
    df_total
    .groupby(["Año", "Mes"], as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
)

ventas_corporativas["Empresa"] = "CORPORATIVO"

ventas_corporativas["Precio_Promedio_Global"] = (
    ventas_corporativas["Total"] / ventas_corporativas["Cantidad"]
)

ventas_corporativas = ventas_corporativas[
    ["Empresa", "Año", "Mes", "Total", "Cantidad", "Precio_Promedio_Global"]
]

ventas_corporativas

,Empresa,Año,Mes,Total,Cantidad,Precio_Promedio_Global
0,CORPORATIVO,2025,agosto 2025,51312.08,787.0,65.199593
1,CORPORATIVO,2025,diciembre 2025,195085.06,57711.0,3.380379
2,CORPORATIVO,2025,julio 2025,64494.73,243.0,265.410412
3,CORPORATIVO,2025,junio 2025,16993.88,224.0,75.865536
4,CORPORATIVO,2025,mayo 2025,257.20,8.0,32.150000
5,CORPORATIVO,2025,noviembre 2025,28729.46,998.0,28.787034
6,CORPORATIVO,2025,octubre 2025,26894.56,790.0,34.043747
7,CORPORATIVO,2025,septiembre 2025,50145.12,440.0,113.966182
8,CORPORATIVO,2026,abril 2026,258641.52,66167.0,3.908920
9,CORPORATIVO,2026,agosto 2026,157486.71,15008.0,10.493517


In [13]:
# PARTE 10 — Unir empresa + corporativo

ventas_mes_empresa_final = pd.concat(
    [ventas_mes_empresa, ventas_corporativas],
    ignore_index=True
)

ventas_mes_empresa_final

,Empresa,Año,Mes,Total,Cantidad,Precio_Promedio_Global
0,RECESA,2025,noviembre 2025,4357.12,252.0,17.290159
1,RECESA,2025,octubre 2025,4910.00,378.0,12.989418
2,RECESA,2025,septiembre 2025,700.00,8.0,87.500000
3,RECESA,2026,abril 2026,121814.70,44215.0,2.755054
4,RECESA,2026,agosto 2026,20893.14,855.0,24.436421
5,RECESA,2026,febrero 2026,10115.00,31900.0,0.317085
6,RECESA,2026,junio 2026,5280.92,3553.0,1.486327
7,RECESA,2026,marzo 2026,40038.10,134875.0,0.296853
8,RECESA,2026,mayo 2026,32759.20,8980.0,3.648018
9,RECESA,2026,septiembre 2026,5829.40,167.0,34.906587


In [14]:
# PARTE 11 — Guardar análisis actualizado

with pd.ExcelWriter("Analisis_Redes_RECESA_REDELSA.xlsx", engine="openpyxl") as writer:
    
    df_total.to_excel(writer, sheet_name="Base Maestra", index=False)
    
    resumen_empresa.to_excel(
        writer,
        sheet_name="Resumen Empresa",
        index=False
    )

    ventas_mes_empresa_final.to_excel(
        writer,
        sheet_name="Ventas por Mes",
        index=False
    )

    top_productos_ventas.to_excel(
        writer,
        sheet_name="Top por Ventas",
        index=False
    )

    top_productos_cantidad.to_excel(
        writer,
        sheet_name="Top por Cantidad",
        index=False
    )


print("Archivo actualizado correctamente")

Archivo actualizado correctamente


In [15]:
##A. ¿Cuántos productos únicos tienes?

df_total["Codigo"].nunique()

622

In [16]:
## B. ¿Hay productos duplicados con distinto nombre?

df_total.groupby("Codigo")["Descripcion"].nunique().sort_values(ascending=False).head(20)

Codigo
10011317-M    1
RO5585        1
RO5516212     1
RO5516312     1
RO558112      1
RO5582        1
RO558212      1
RO5583        1
RO5584        1
RO5586        1
RO838112      1
RO57165       1
RO5783        1
RO5784        1
RO5785        1
RO5916312     1
RO8342        1
RO834212      1
RO5516112     1
RO538312      1
Name: Descripcion, dtype: int64

In [17]:
### C. Productos sin ventas

(df_total["Total"] == 0).sum()

0

In [18]:
### Ventas corporativas totales

df_total["Total"].sum()

2068945.0200000003

In [19]:
### Unidades vendidas

df_total["Cantidad"].sum()

410033.0

In [20]:
### Ticket promedio global

df_total["Total"].sum() / df_total["Cantidad"].sum()

5.045801240387969

In [21]:
### A. Top 10 productos por ingresos

top_10_ingresos = (
    df_total
    .groupby(["Codigo", "Descripcion"], as_index=False)["Total"]
    .sum()
    .sort_values("Total", ascending=False)
    .head(10)
)

top_10_ingresos


,Codigo,Descripcion,Total
354,PSM,PERNO S/MUESTRA,684001.18
180,BRI-25355,BARRA LISA F1554/GR55 25MM X 3.55 MTS,45865.17
441,ROA325582,"TORNILLO A325 5/8 X 2""",41102.00
607,TROA32558,TUERCA HEX. A325 5/8,39775.50
43,21032028-M,TOYOTA HILUX MOD TRA 3a 20X28 REFUERZO,36716.96
601,TROA3251,"TUERCA HEX. A325 1""",35107.69
606,TROA32534,TUERCA HEX. A325 3/4,35009.18
442,ROA32558212,TORNILLO A325 5/8 X 2 1/2,33528.32
374,RC-55-148,RESORTE C INTER DEL. 55-148 9HN 4X7/16 (S4000),33333.50
434,ROA3251312,TORNILLO HEX. A325 1 X 3 1/2,33141.46


In [22]:
### B. Top 10 productos por demanda

top_10_demanda = (
    df_total
    .groupby(["Codigo", "Descripcion"], as_index=False)["Cantidad"]
    .sum()
    .sort_values("Cantidad", ascending=False)
    .head(10)
)

top_10_demanda

,Codigo,Descripcion,Cantidad
578,TPL14112,TORNILLO LAMINA 14 X 1 1/2,134520.0
464,RPF43658,ROLDANA F436 5/8,28446.0
607,TROA32558,TUERCA HEX. A325 5/8,26150.0
610,TSDT141,TORNILLO POLSER P/BROCA TROPICALIZADO1/4 X 1,24900.0
354,PSM,PERNO S/MUESTRA,12074.0
606,TROA32534,TUERCA HEX. A325 3/4,11072.0
441,ROA325582,"TORNILLO A325 5/8 X 2""",10580.0
604,TROA32512,TUERCA HEX. A325 1/2,10143.0
461,RPF43612,"ROLDANA F436 1/2""",10063.0
453,RP38,ROLDANA 3/8,8974.0


In [23]:
### Evolución mensual

ventas_mes = (
    df_total
    .groupby("Mes", as_index=False)["Total"]
    .sum()
)

ventas_mes

,Mes,Total
0,abril 2026,258641.52
1,agosto 2025,51312.08
2,agosto 2026,157486.71
3,diciembre 2025,195085.06
4,enero 2026,137408.91
5,febrero 2026,190801.80
6,julio 2025,64494.73
7,julio 2026,165908.77
8,junio 2025,16993.88
9,junio 2026,204283.21


In [24]:

ventas_mes = ventas_mes.reset_index(drop=True)

ventas_mes.index = ventas_mes.index + 1

ventas_mes

,Mes,Total
1,abril 2026,258641.52
2,agosto 2025,51312.08
3,agosto 2026,157486.71
4,diciembre 2025,195085.06
5,enero 2026,137408.91
6,febrero 2026,190801.80
7,julio 2025,64494.73
8,julio 2026,165908.77
9,junio 2025,16993.88
10,junio 2026,204283.21


In [25]:
df_total.to_excel(
    "Base_Maestra_Redes.xlsx",
    index=False
)

print("Base exportada correctamente")

Base exportada correctamente


In [26]:
import pandas as pd

df_total = pd.read_excel("Base_Maestra_Redes.xlsx")

MESES_ES = {
    1: "enero",
    2: "febrero",
    3: "marzo",
    4: "abril",
    5: "mayo",
    6: "junio",
    7: "julio",
    8: "agosto",
    9: "septiembre",
    10: "octubre",
    11: "noviembre",
    12: "diciembre"
}

# Limpiar Mes
df_total["Mes"] = (
    df_total["Mes"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# Limpiar códigos
df_total["Codigo"] = (
    df_total["Codigo"]
    .astype("string")
    .fillna("SIN CODIGO")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# Limpiar descripciones
df_total["Descripcion"] = (
    df_total["Descripcion"]
    .astype("string")
    .fillna("SIN DESCRIPCION")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

# Limpiar Total
df_total["Total"] = (
    df_total["Total"]
    .astype(str)
    .str.replace("Q", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_total["Total"] = pd.to_numeric(
    df_total["Total"],
    errors="coerce"
).fillna(0)

# Limpiar Cantidad
df_total["Cantidad"] = (
    df_total["Cantidad"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df_total["Cantidad"] = pd.to_numeric(
    df_total["Cantidad"],
    errors="coerce"
).fillna(0)


def generar_reporte(df_total, mes_num=7, anio=2026):

    mes_nombre = MESES_ES[mes_num]
    mes_reporte = f"{mes_nombre} {anio}"

    meses_acumulado = [
        f"{MESES_ES[mes]} {anio}"
        for mes in range(1, mes_num)
    ]

    mes_inicio_nombre = MESES_ES[1].capitalize()
    mes_fin_nombre = MESES_ES[mes_num - 1].capitalize()
    mes_actual_cap = mes_nombre.capitalize()

    col_acum_q = f"Acumulado {mes_inicio_nombre} a {mes_fin_nombre} (Q)"
    col_acum_u = f"Acumulado {mes_inicio_nombre} a {mes_fin_nombre} Unidades"
    col_mes_q = f"{mes_actual_cap} Ventas (Q)"
    col_mes_u = f"{mes_actual_cap} Unidades"

    acumulado = (
        df_total[df_total["Mes"].isin(meses_acumulado)]
        .groupby(
            ["Codigo", "Descripcion"],
            as_index=False,
            dropna=False
        )
        .agg(
            Total=("Total", "sum"),
            Cantidad=("Cantidad", "sum")
        )
        .rename(columns={
            "Total": col_acum_q,
            "Cantidad": col_acum_u
        })
    )

    mes_actual = (
        df_total[df_total["Mes"].eq(mes_reporte)]
        .groupby(
            ["Codigo", "Descripcion"],
            as_index=False,
            dropna=False
        )
        .agg(
            Total=("Total", "sum"),
            Cantidad=("Cantidad", "sum")
        )
        .rename(columns={
            "Total": col_mes_q,
            "Cantidad": col_mes_u
        })
    )

    reporte = pd.merge(
        acumulado,
        mes_actual,
        on=["Codigo", "Descripcion"],
        how="outer"
    )

    columnas_numericas = [
        col_acum_q,
        col_acum_u,
        col_mes_q,
        col_mes_u
    ]

    reporte[columnas_numericas] = (
        reporte[columnas_numericas].fillna(0)
    )

    reporte["Total Ventas (Q)"] = (
        reporte[col_acum_q] + reporte[col_mes_q]
    )

    reporte["Total Unidades"] = (
        reporte[col_acum_u] + reporte[col_mes_u]
    )

    reporte["Precio Promedio (Q)"] = (
        reporte["Total Ventas (Q)"]
        .div(reporte["Total Unidades"])
        .replace([float("inf"), -float("inf")], 0)
        .fillna(0)
    )

    reporte = reporte.sort_values(
        "Total Ventas (Q)",
        ascending=False
    ).reset_index(drop=True)

    return reporte, mes_actual_cap, anio


reporte, mes_actual_cap, anio = generar_reporte(
    df_total,
    mes_num=7,
    anio=2026
)

# Validación
meses_enero_julio = [
    f"{MESES_ES[mes]} 2026"
    for mes in range(1, 8)
]

total_base = df_total.loc[
    df_total["Mes"].isin(meses_enero_julio),
    "Total"
].sum()

total_reporte = reporte["Total Ventas (Q)"].sum()

print(f"Total directo de la base: Q {total_base:,.2f}")
print(f"Total generado en reporte: Q {total_reporte:,.2f}")
print(f"Diferencia: Q {total_base - total_reporte:,.2f}")

reporte.to_excel(
    "Reporte_Ventas_Redes_Julio_2026.xlsx",
    index=False
)

Total directo de la base: Q 1,266,589.33
Total generado en reporte: Q 1,266,589.33
Diferencia: Q -0.00


In [27]:
import pandas as pd

MESES_ES = {
    1: "enero",
    2: "febrero",
    3: "marzo",
    4: "abril",
    5: "mayo",
    6: "junio",
    7: "julio",
    8: "agosto",
    9: "septiembre",
    10: "octubre",
    11: "noviembre",
    12: "diciembre"
}


def generar_reporte(
    df_total,
    mes_num=8,
    anio=2026,
    mes_inicio=5,
    anio_inicio=2025
):
    """
    Genera un reporte con:

    - Acumulado desde mayo 2025 hasta julio 2026.
    - Ventas únicamente de agosto 2026.
    - Total acumulado más agosto 2026.
    """

    df_total = df_total.copy()

    # Normalizar la columna Mes
    df_total["Mes"] = (
        df_total["Mes"]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

    # Convertir columnas numéricas
    df_total["Total"] = pd.to_numeric(
        df_total["Total"],
        errors="coerce"
    ).fillna(0)

    df_total["Cantidad"] = pd.to_numeric(
        df_total["Cantidad"],
        errors="coerce"
    ).fillna(0)

    # Periodo del mes que se reportará: agosto 2026
    periodo_reporte = pd.Period(
        year=anio,
        month=mes_num,
        freq="M"
    )

    # Inicio del acumulado: mayo 2025
    periodo_inicio = pd.Period(
        year=anio_inicio,
        month=mes_inicio,
        freq="M"
    )

    # Fin del acumulado: mes anterior al reporte, julio 2026
    periodo_fin = periodo_reporte - 1

    if periodo_inicio > periodo_fin:
        raise ValueError(
            "El inicio del acumulado no puede ser posterior "
            "al mes anterior al reporte."
        )

    mes_nombre = MESES_ES[mes_num]
    mes_reporte = f"{mes_nombre} {anio}"

    # Crear todos los meses desde mayo 2025 hasta julio 2026
    periodos_acumulado = pd.period_range(
        start=periodo_inicio,
        end=periodo_fin,
        freq="M"
    )

    meses_acumulado = [
        f"{MESES_ES[periodo.month]} {periodo.year}"
        for periodo in periodos_acumulado
    ]

    inicio_etiqueta = (
        f"{MESES_ES[periodo_inicio.month].capitalize()} "
        f"{periodo_inicio.year}"
    )

    fin_etiqueta = (
        f"{MESES_ES[periodo_fin.month].capitalize()} "
        f"{periodo_fin.year}"
    )

    mes_actual_cap = mes_nombre.capitalize()

    col_acum_q = (
        f"Acumulado {inicio_etiqueta} a {fin_etiqueta} (Q)"
    )

    col_acum_u = (
        f"Acumulado {inicio_etiqueta} a {fin_etiqueta} Unidades"
    )

    col_mes_q = f"{mes_actual_cap} {anio} Ventas (Q)"
    col_mes_u = f"{mes_actual_cap} {anio} Unidades"

    # Acumulado mayo 2025 a julio 2026
    acumulado = (
        df_total.loc[
            df_total["Mes"].isin(meses_acumulado)
        ]
        .groupby(
            ["Codigo", "Descripcion"],
            as_index=False,
            dropna=False
        )
        .agg(
            Total=("Total", "sum"),
            Cantidad=("Cantidad", "sum")
        )
        .rename(
            columns={
                "Total": col_acum_q,
                "Cantidad": col_acum_u
            }
        )
    )

    # Ventas únicamente de agosto 2026
    mes_actual = (
        df_total.loc[
            df_total["Mes"].eq(mes_reporte)
        ]
        .groupby(
            ["Codigo", "Descripcion"],
            as_index=False,
            dropna=False
        )
        .agg(
            Total=("Total", "sum"),
            Cantidad=("Cantidad", "sum")
        )
        .rename(
            columns={
                "Total": col_mes_q,
                "Cantidad": col_mes_u
            }
        )
    )

    # Unir acumulado y agosto
    reporte = pd.merge(
        acumulado,
        mes_actual,
        on=["Codigo", "Descripcion"],
        how="outer"
    )

    columnas_numericas = [
        col_acum_q,
        col_acum_u,
        col_mes_q,
        col_mes_u
    ]

    reporte[columnas_numericas] = (
        reporte[columnas_numericas].fillna(0)
    )

    # Total mayo 2025 a agosto 2026
    reporte["Total Ventas (Q)"] = (
        reporte[col_acum_q] + reporte[col_mes_q]
    )

    reporte["Total Unidades"] = (
        reporte[col_acum_u] + reporte[col_mes_u]
    )

    reporte["Precio Promedio (Q)"] = (
        reporte["Total Ventas (Q)"]
        .div(reporte["Total Unidades"])
        .replace([float("inf"), -float("inf")], 0)
        .fillna(0)
    )

    reporte = (
        reporte[
            [
                "Codigo",
                "Descripcion",
                col_acum_q,
                col_acum_u,
                col_mes_q,
                col_mes_u,
                "Total Ventas (Q)",
                "Total Unidades",
                "Precio Promedio (Q)"
            ]
        ]
        .sort_values(
            "Total Ventas (Q)",
            ascending=False
        )
        .reset_index(drop=True)
    )

    return reporte, mes_actual_cap, anio, meses_acumulado


df_total = pd.read_excel("Base_Maestra_Redes.xlsx")

reporte, mes_actual_cap, anio, meses_acumulado = generar_reporte(
    df_total,
    mes_num=8,
    anio=2026,
    mes_inicio=5,
    anio_inicio=2025
)

nombre_archivo = (
    f"Reporte_Ventas_Redes_{mes_actual_cap}_{anio}.xlsx"
)

reporte.to_excel(
    nombre_archivo,
    index=False
)

print("Meses incluidos en el acumulado:")
print(meses_acumulado)

print(
    f"Reporte de {mes_actual_cap} {anio} creado correctamente: "
    f"{nombre_archivo}"
)

Meses incluidos en el acumulado:
['mayo 2025', 'junio 2025', 'julio 2025', 'agosto 2025', 'septiembre 2025', 'octubre 2025', 'noviembre 2025', 'diciembre 2025', 'enero 2026', 'febrero 2026', 'marzo 2026', 'abril 2026', 'mayo 2026', 'junio 2026', 'julio 2026']
Reporte de Agosto 2026 creado correctamente: Reporte_Ventas_Redes_Agosto_2026.xlsx


In [28]:
import pandas as pd

df_total = pd.read_excel("Base_Maestra_Redes.xlsx")

In [29]:
meses_num = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

df_total["Mes_Texto"] = df_total["Mes"]

df_total["Numero_Mes"] = (
    df_total["Mes"]
    .str.extract(r"([a-zA-Záéíóúñ]+)")[0]
    .str.lower()
    .map(meses_num)
)

df_total["Año"] = (
    df_total["Mes"]
    .str.extract(r"(\d{4})")
    .astype(int)
)

df_total["Orden_Mes"] = (
    df_total["Año"] * 100 +
    df_total["Numero_Mes"]
)

In [30]:
df_recesa_raw.head(5)
df_redelsa_raw.head(5)

,0,1,2,3,4,5,6,7,8,9,...,45,46,47,48,49,50,51,52,53,54
0,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,mayo 2025,NaN,NaN,junio 2025,NaN,NaN,julio 2025,NaN,NaN,...,NaN,agosto 2026,NaN,NaN,septiembre 2026,NaN,NaN,NaN,NaN,NaN
2,NaN,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,...,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio
3,Total,257.2,8,28.70625,16993.88,224,67.737098,64494.73,243,236.973498,...,5.785758,136593.57,14153,8.617155,205127.49,19198,9.540035,1822247.44,184850,8.801767
4,PRODUCTO TERMINADO / BALANCIN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,571.14,2,254.975


In [31]:
df_recesa = transformar_redes(df_recesa_raw, "RECESA")
df_redelsa = transformar_redes(df_redelsa_raw, "REDELSA")

df_total = pd.concat([df_recesa, df_redelsa], ignore_index=True)

meses_num = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

df_total["Mes_Texto"] = df_total["Mes"]

df_total["Numero_Mes"] = (
    df_total["Mes"]
    .str.extract(r"([a-zA-Záéíóúñ]+)")[0]
    .str.lower()
    .map(meses_num)
)

df_total["Año"] = df_total["Mes"].str.extract(r"(\d{4})").astype(int)

df_total["Orden_Mes"] = df_total["Año"] * 100 + df_total["Numero_Mes"]

df_total.to_excel("Base_Maestra_Redes.xlsx", index=False)

print("Base actualizada con Mes_Texto y Orden_Mes")